In [2]:
import cv2
import os
from ultralytics import YOLO

# 1. Load your best model
model = YOLO('../best_yolov8_coral_reef/runs/detect/reef_coral/weights/best.pt')

# Folder Setup
video_folder = '../yolov8_model/videos' # <-- Update this to your folder path
video_extensions = ('.mp4', '.avi', '.mov', '.mkv')

# Configuration
SLOT_CAPACITIES = [3, 1, 2, 2, 1, 3]
NUM_SLOTS = len(SLOT_CAPACITIES)
STABILITY_THRESHOLD = 10 

# Get list of videos
video_files = sorted([f for f in os.listdir(video_folder) if f.lower().endswith(video_extensions)])

for video_name in video_files:
    video_path = os.path.join(video_folder, video_name)
    cap = cv2.VideoCapture(video_path)
    
    if not cap.isOpened():
        print(f"Skipping {video_name}: Could not open.")
        continue

    # Video properties
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps    = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    # Timing logic
    current_frame_idx = 0
    cutoff_frame = total_frames - int(49 * fps)
    if cutoff_frame <= 0: 
        cutoff_frame = total_frames

    # Crop setup (Bottom 2/5ths)
    crop_h = int(height * (2/5))
    start_y = height - crop_h

    # Tracking & Scoring State (Reset for every video)
    reef_grids = [[0] * NUM_SLOTS for _ in range(5)]
    claimed_coral_ids = set()
    potential_scores = {}    
    scoring_log = [] 
    total_points = 0

    print(f"\n>>> Processing: {video_name} ({total_frames} frames)")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret or current_frame_idx >= cutoff_frame:
            break

        # Apply Crop
        cropped = frame[start_y:height, 0:width]
        
        # Track
        results = model.track(cropped, persist=True, tracker="botsort.yaml", conf=0.25, verbose=False)

        if results[0].boxes.id is not None:
            boxes = results[0].boxes.xyxy.cpu().numpy()
            clss = results[0].boxes.cls.cpu().numpy().astype(int)
            ids = results[0].boxes.id.cpu().numpy().astype(int)

            # --- ORIGINAL REEF SORTING (Left to Right) ---
            frame_reefs = [boxes[i] for i, c in enumerate(clss) if model.names[c] == 'reef']
            frame_reefs.sort(key=lambda x: x[0])

            current_frame_coral_ids = set()

            for i, c_id in enumerate(ids):
                if model.names[clss[i]] == 'coral':
                    if c_id in claimed_coral_ids:
                        continue
                    
                    current_frame_coral_ids.add(c_id)
                    cx, cy = (boxes[i][0] + boxes[i][2])/2, (boxes[i][1] + boxes[i][3])/2
                    
                    in_scoring_zone = False
                    for r_idx, r_box in enumerate(frame_reefs):
                        rx1, ry1, rx2, ry2 = r_box
                        rw, rh = rx2 - rx1, ry2 - ry1
                        
                        # Scoring Zone: Top 20% of reef height
                        if (rx1 < cx < rx2) and (ry1 < cy < ry1 + (rh * 0.20)):
                            in_scoring_zone = True
                            rel_x = cx - rx1
                            slot_idx = int((rel_x / rw) * NUM_SLOTS)
                            slot_idx = max(0, min(slot_idx, NUM_SLOTS - 1))

                            # Stability Logic
                            p = potential_scores.get(c_id, {"reef": r_idx, "slot": slot_idx, "count": 0})
                            if p["reef"] == r_idx and p["slot"] == slot_idx:
                                p["count"] += 1
                            else:
                                p = {"reef": r_idx, "slot": slot_idx, "count": 1}
                            potential_scores[c_id] = p

                            if p["count"] >= STABILITY_THRESHOLD:
                                max_allowed = SLOT_CAPACITIES[slot_idx]
                                if r_idx < len(reef_grids) and reef_grids[r_idx][slot_idx] < max_allowed:
                                    # LOG SCORE
                                    seconds = current_frame_idx / fps
                                    timestamp = f"{int(seconds // 60):02d}:{int(seconds % 60):02d}"
                                    
                                    reef_grids[r_idx][slot_idx] += 1
                                    claimed_coral_ids.add(c_id)
                                    total_points += 4
                                    
                                    scoring_log.append((timestamp, r_idx, slot_idx))
                                    print(f"  [{timestamp}] Score! Reef {r_idx} Slot {slot_idx} (Points: {total_points})")
                                    
                                    del potential_scores[c_id]
                            break 
                    
                    if not in_scoring_zone and c_id in potential_scores:
                        del potential_scores[c_id]

            # Clean up disappeared IDs
            for pid in list(potential_scores.keys()):
                if pid not in current_frame_coral_ids:
                    del potential_scores[pid]

        current_frame_idx += 1

    cap.release()

    # --- FINAL VIDEO SUMMARY ---
    print(f"--- Finished: {video_name} | Final Score: {total_points} ---")
    print("-" * 50)

print("\nAll videos in folder have been processed.")


>>> Processing: Final 1 - 2025 Central Missouri Regional.mp4 (5769 frames)
  [00:11] Score! Reef 0 Slot 0 (Points: 4)
  [00:12] Score! Reef 1 Slot 0 (Points: 8)
  [00:17] Score! Reef 1 Slot 2 (Points: 12)
  [00:18] Score! Reef 0 Slot 0 (Points: 16)
  [00:23] Score! Reef 1 Slot 3 (Points: 20)
  [00:25] Score! Reef 1 Slot 2 (Points: 24)
  [00:26] Score! Reef 0 Slot 2 (Points: 28)
  [00:28] Score! Reef 0 Slot 2 (Points: 32)
  [00:31] Score! Reef 1 Slot 4 (Points: 36)
  [00:38] Score! Reef 0 Slot 0 (Points: 40)
  [00:38] Score! Reef 0 Slot 1 (Points: 44)
  [00:44] Score! Reef 1 Slot 3 (Points: 48)
  [01:07] Score! Reef 1 Slot 5 (Points: 52)
  [01:14] Score! Reef 1 Slot 5 (Points: 56)
  [01:14] Score! Reef 1 Slot 5 (Points: 60)
  [01:18] Score! Reef 0 Slot 3 (Points: 64)
  [01:35] Score! Reef 0 Slot 3 (Points: 68)
  [01:39] Score! Reef 1 Slot 0 (Points: 72)
  [01:39] Score! Reef 1 Slot 0 (Points: 76)
  [01:54] Score! Reef 1 Slot 1 (Points: 80)
  [02:18] Score! Reef 0 Slot 4 (Points: 84)
--